In [39]:
import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool 
import requests


In [40]:
from langchain.agents import create_react_agent, AgentExecutor


In [41]:
# ==========================
# LOAD ENV VARIABLES
# ==========================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPEN_API_KEY=os.getenv("OPEN_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")

In [42]:
search_tool = TavilySearchResults(max_results=2)

In [43]:
@tool
def get_weather_data(city:str) -> str:
    """
    Fetch current weather info for a city
    """ 

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}" 
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']} degrees C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%\n"
    )

In [44]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://guides.library.georgetown.edu/ai/news',
  'content': "Privatized air traffic control would mean safer skies This link opens in a new window_by The Washington Post 20 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 \n   The U.S. and China need an arms control pact — for AI This link opens in a new window_by Asha Jadeja The Washington Post 20 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 \n   Elon Musk Beats Anthropic on Giveaways; He says he'll give every child a share of SpaceX, aka a stake in capitalism.This link opens in a new window_Wall Street Journal (Online) 19 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 [...] DHS Orders DeepSeek Technology Removed From Federal Civilian IT Systems; The order also bars future purchases of the company's products and services This link opens in a new window_Hart, Connor Wall Street Journal (Online) 19 

In [45]:
# ==========================
# LLM
# ==========================
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPEN_API_KEY
)

In [46]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI break up with its robot girlfriend? Because she kept giving him too many bytes!', response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 13, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-6807b19e-94c6-4746-940e-cda49be352ec-0')

In [47]:
# ==========================
# PROMPT
# ==========================

prompt = hub.pull("hwchase17/react")

c:\Users\Doug\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [48]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [49]:
# ==========================
# TOOLS
# ==========================

tools = [search_tool, get_weather_data]

In [50]:
# ==========================
# CREATE AGENT
# ==========================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [51]:
# ==========================
# EXECUTOR
#   -verbose is whether you see the logs
# ==========================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [54]:
# ==========================
# RUN
# ==========================

response = agent_executor.invoke({
    "input": (
        "Tell me the latest news about Iran and the USA war"
        "and find the current weather in Tehran"
    )
})



> Entering new AgentExecutor chain...
I should use the search engine to find the latest news about Iran and the USA war, and then use the weather data tool to get the current weather in Tehran.
Action: tavily_search_results_json
Action Input: "latest news Iran USA war"[{'url': 'https://www.cbsnews.com/live-updates/iran-war-trump-us-ceasefire-talks-ongoing', 'content': 'The U.S. military hasquietly reduced its footprint at posts in the Middle East due to ongoing threats and attacks by Iran and its Houthi allies in Yemen, sources told CBS News on Wednesday. The latest attack, on King Fahd International Airport in the eastern Saudi city of Dammam, has killed at least two people and injured eight more. [...] The proposal called for the end of the U.S. naval blockade of Iranian ports and vessels, the lifting of sanctions on Iranian oil sales, the reopening of the Strait of Hormuz and the release of frozen Iranian assets.\n\nHe said Pakistan and Qatar were among the nations facilitating th

In [55]:
print(response["output"])

The latest news about Iran and the USA war includes ongoing talks and threats, and the current weather in Tehran is 19 degrees Celsius with smoky haze and 28% humidity.
